In [3]:
import os
import requests
import json
from dotenv import load_dotenv
import requests_cache
import re
import time
import random
import pandas as pd

In [4]:
# load .env file, get API key
#load_dotenv()
#API_KEY = os.getenv("api_key")

# make sure API key is available
#if not API_KEY:
    #raise ValueError("no API key found")

# set up a cached session
session = requests_cache.CachedSession('output/steam_api_cache', expire_after=86400)

In [5]:
# sample output
def fetch_steam_app_data(app_id: int) -> dict:
    """
    Fetch detailed information about a Steam application using the Storefront API.
    """
    
    # define the target API endpoint
    url = "https://store.steampowered.com/api/appdetails"
    
    # construct query parameters for the GET request
    params = {
        "appids": app_id,
        "cc": "us",      # currency: USD
        "l": "english"   # language: English
    }
    
    try:

        # use cached session to sent request, timeout after 10 seconds
        response = session.get(url, params=params, timeout=10)
        response.raise_for_status()
        
        # parse the JSON response into a json
        raw_data = response.json()
        
        # check if the response contains valid data for the given app_id
        app_id_str = str(app_id)
        if app_id_str in raw_data and raw_data[app_id_str].get("success"):
            return raw_data[app_id_str]["data"]
        else:
            print(f"No valid data found for AppId:{app_id_str}")
            return None
            
    except requests.exceptions.RequestException as e:
        print(f"HTTP Request failed: {e}")
        return None


In [6]:
sample_output = fetch_steam_app_data(1091500)

with open("output/sample_app_data.json", "w") as f:
    json.dump(sample_output, f, indent=4)

In [8]:
# 1) 从 Steam 商店搜索页抓一批 appid
#    filter="topsellers" 可换成 "newandtrending" / "specials"
def get_appids(target=2000, pages=120, flt="topsellers"):
    s = requests.Session()
    s.headers.update({"User-Agent": "Mozilla/5.0"}) 

    appids, seen = [], set()
    for p in range(1, pages + 1):
        html = s.get(
            "https://store.steampowered.com/search/",
            params={"filter": flt, "page": p},
            timeout=30
        ).text

        # 搜索页里每个游戏条目都有 data-ds-appid="1091500"
        # 有时会是 "123,456"（例如包含多个 appid），所以后面要 split(",")
        for x in re.findall(r'data-ds-appid="([\d,]+)"', html):
            for a in x.split(","):
                a = int(a)
                if a not in seen:
                    seen.add(a)
                    appids.append(a)
                    if len(appids) >= target:
                        return appids

        # 翻页限速
        time.sleep(0.5 + random.random() * 0.3)

    return appids

# 2) 429 限流自动重试：指数退避（1.2s, 2.4s, 4.8s...）+ 随机抖动
def safe_fetch(appid, tries=7, base=1.2):
    for i in range(tries):
        try:
            return fetch_steam_app_data(appid)
        except Exception as e:
            # 只对 429 处理，其它错误就跳过该游戏
            if "429" not in str(e):
                return None

            wait = base * (2 ** i) + random.random()
            print(f"[429] appid={appid} wait={wait:.1f}s (try {i+1}/{tries})")
            time.sleep(60) # sleep longer instead of wait
    return None


# 3) 主循环：抓 1000 个游戏，输出 JSONL（每行一个 sample.json 结构）
TARGET = 1000
OUT = "steam_1000.jsonl"

appids = get_appids(target=TARGET * 2, pages=120, flt="topsellers")
random.shuffle(appids)  # 打乱顺序，避免全部都是榜单前排类型

n = 0
with open(OUT, "w", encoding="utf-8") as f:
    for appid in appids:
        d = safe_fetch(appid)

        # 只保留真正的游戏（过滤 demo / dlc / application）
        if d and (d.get("type") or "").lower() == "game":
            f.write(json.dumps(d, ensure_ascii=False) + "\n") 
            n += 1
            if n >= TARGET:
                break

        time.sleep(1.0 + random.random() * 0.5)

print(f"Done. saved={n} -> {OUT}")

KeyboardInterrupt: 

In [ ]:
# convert the steam_1000.jsonl to dataframe
def clean_html(text):
    if not isinstance(text, str):
        return ""
    # remove HTML tags like <>, <b>, etc.
    return re.sub(r'<.*?>', '', text)

def flatten_list(items):
    # this is to extract items from the internal list as new columns 
    if isinstance(items, list):
        # extract the description field and join with commas 
        return ", ".join(item.get("description", "") for item in items)
    return ""

raw_data = []
# load the JSONL file into a list
with open("steam_1000.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        raw_data.append(json.loads(line))

# convert to DataFrame, flattening the lists
df = pd.json_normalize(raw_data)

# deal with complex list features 
df['genres(cleaned)'] = df['genres'].apply(flatten_list)
df['categories(cleaned)'] = df['categories'].apply(flatten_list)
df['legal_notice(cleaned)'] = df['legal_notice'].apply(flatten_list)


# clean HTML from description column
text_cols = ['detailed_description', 'short_description', 'about_the_game', 'supported_languages',
             'pc_requirements.minimum', 'pc_requirements.recommended', 
             'mac_requirements.minimum', 'mac_requirements.recommended',
             'linux_requirements.minimum', 'linux_requirements.recommended']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].apply(clean_html)

# price in the original data is represented in cents
# convert it to get the dollar format, NaN remains NaN in this step
if 'price_overview.initial' in df.columns:
    df['price_overview.initial'] = df['price_overview.initial'] / 100

if 'price_overview.final' in df.columns:
    df['price_overview.final'] = df['price_overview.final'] / 100

# summarize and select(rename) the cleaned columns we need to form the final data frame
final_columns = ['name', 'steam_appid', 'type', 'is_free', 'developers', 'publishers', 
                 'detailed_description', 'short_description', 'about_the_game', 'supported_languages',
                 'pc_requirements.minimum', 'pc_requirements.recommended', 
                 'mac_requirements.minimum', 'mac_requirements.recommended',
                 'linux_requirements.minimum', 'linux_requirements.recommended',
                 'genres(cleaned)', 'categories(cleaned)',
                 'price_overview.currency', 'price_overview.initial', 'price_overview.final', 'price_overview.discount_percent', 
                 'platforms.windows', 'platforms.mac', 'platforms.linux', 
                 'packages', 'recommendations.total', 'release_date.date', 'legal_notice(cleaned)'
    ]

df_final = pd.DataFrame(df[[c for c in final_columns if c in df.columns]])
print("Final DataFrame shape:", df_final.shape)
print("\nThe first 5 rows of the final DataFrame:")
print(df_final.head())


Final DataFrame shape: (1000, 29)

The first 5 rows of the final DataFrame:
                         name  steam_appid  type  is_free  \
0  Tom Clancy’s The Division™       365590  game    False   
1            Counter-Strike 2          730  game     True   
2                    Megabonk      3405340  game    False   
3             Resident Evil 4      2050650  game    False   
4              Sector Unknown      2734270  game    False   

                       developers                      publishers  \
0         [Massive Entertainment]                       [Ubisoft]   
1                         [Valve]                         [Valve]   
2                       [vedinad]                       [vedinad]   
3              [CAPCOM Co., Ltd.]              [CAPCOM Co., Ltd.]   
4  [Creative Storm Entertainment]  [Creative Storm Entertainment]   

                                detailed_description  \
0  Comparison GridDefinitive EditionGet the compl...   
1  For over two decades, Count

In [29]:
# get the dataset summary information
print("Dataset Summary:")
print(df_final.info())

Dataset Summary:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 29 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   name                             1000 non-null   object 
 1   steam_appid                      1000 non-null   int64  
 2   type                             1000 non-null   object 
 3   is_free                          1000 non-null   bool   
 4   developers                       1000 non-null   object 
 5   publishers                       999 non-null    object 
 6   detailed_description             1000 non-null   object 
 7   short_description                1000 non-null   object 
 8   about_the_game                   1000 non-null   object 
 9   supported_languages              1000 non-null   object 
 10  pc_requirements.minimum          1000 non-null   object 
 11  pc_requirements.recommended      1000 non-null   object 
 12  mac_

In [30]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
}

 ### Question 2:  
 How do textual features and player sentiments in game reviews differ between high-selling and low-selling games, and what are the key drivers of positive and negative feedback for games?  
 
 Specifically, we will use TF-IDF to analyze the text of the reviews and also perform a comparative analysis of the differences and sentiment drivers in top-sellers vs. low-sellers games. 


In [1]:
# fetch the first 100 reviews for a given steam appid
# more than 100 reviews might get banned
def fetch_reviews(appid, num_per_page=100):
    url = f"https://store.steampowered.com/appreviews/{appid}"
    params = {
        'json': 1, # means return json format
        'filter': 'all', # get all the historical reviews (not only recent), usually ordered by the most valued reviews
        'language': 'english', # only get the English reviews
        'num_reviews': num_per_page, # set the max to 100 reviews 
        'day_range': 9223372036854775807, # this is the maximum value of 64-bit signed integer(standard representation for all time) to get all the reviews from the past
        'review_type': 'all', # get all types of reviews (including positive and negative)
        'purchase_type': 'all', # get all types of purchased reviews
        'cursor': '*' # * means start from the beginning
    }
    try:
        response = requests.get(url=url, params=params, headers=headers, timeout=20)

        if response.status_code == 429:
            print(f"Rate limited on appid: {appid}, sleeping...")
            time.sleep(60) # sleep for 60 seconds
            return fetch_reviews(appid, num_per_page) # retry the request
        
        data = response.json()
        if data.get("success") and "reviews" in data:
            return data["reviews"]
        return None
    
    except Exception as e:
        print(f"Error fetching reviews for appid: {appid}, error: {e}")
        return None
    
# get the appids from df_final
appids = df_final['steam_appid'].unique().tolist()

out_file = "reviews_1000.jsonl"
count = 0

# iterate over each appid and fetch the first 100 reviews for this game
# store them into the reviews_1000 json file
with open(out_file, "w", encoding="utf-8") as f:
    for appid in appids:
        reviews = fetch_reviews(appid)

        if reviews:
            for r in reviews:
                # save each review as a separate line with its appid
                clean_text = r.get('review', '').replace('\n', ' ').replace('\r', ' ').strip()

                review = {
                    "appid": int(appid),
                    # get the review text
                    "review": clean_text,
                    # get the information on whether the reviewer recommended or not (true or false)
                    'voted_up': r.get('voted_up'),
                    # get the number of upvotes
                    # means how many people think this reviews helpful, consider increasing the weight of the review
                    'votes_up': r.get('votes_up'), 
                    # get the number of downvotes, means how many people think this reviews not helpful
                    'votes_down': r.get('votes_down'),
                    # get the playtime at review, the number of hours that the reviewer play this game
                    'playtime_at_review': r.get('author', {}).get('playtime_at_review'), 
                    # total playtime of this reviewer until now
                    'playtime_forever': r.get('author', {}).get('playtime_forever'), 
                    # bool: whether the reviewer received the game for free
                    'received_for_free': r.get('received_for_free'), 
                    'comment_count': r.get('comment_count'), 
                    # a credibility score between 0 and 1, calulated based on the votes_up and publish time
                    'weighted_vote_score': r.get('weighted_vote_score')
                }
                f.write(json.dumps(review) + "\n")          
            count += 1
            if count % 50 == 0:
                print(f"Processed and saved {count}/{len(appids)} games.")

        # slow down the fetch, use random to avoid being identify as robot, avoid blocked
        # approx 3-4 seconds of sleep
        time.sleep(3+random.random())

print(f"Finished processing all the reviews for games. Total saved: {count}")

NameError: name 'df_final' is not defined